# PyLBLRTM — prueba del forward model

Flujo completo: `from_hapi` → `SpectralData` → `AtmosphericProfile` → transmitancia LBL.

Dos APIs disponibles:
- **Opción A** — `run_forward(sd, profile, "forward_config.yaml")` : editas el YAML y listo.
- **Opción B** — `compute_transmittance(sd, profile, ...)` : control directo por argumentos Python.

In [ ]:
from pylblrtm import (
    from_hapi,
    us_standard_atmosphere,
    load_config,
    run_forward,
    compute_transmittance,
)

## 1. Datos espectrales

Descarga lineas de HITRAN (o lee la cache local) y construye un `SpectralData`.
Cambia `iso_id` a `[1, 2]` o `"all"` para incluir varios isotopologos.

In [ ]:
sd = from_hapi("CO2", nu_min=700, nu_max=750, iso_id=1)
print(sd)

## 2. Perfil atmosferico

`us_standard_atmosphere()` devuelve un `AtmosphericProfile` con perfiles USSA76 interpolados
a 20 capas entre superficie y ~47 km.  
Puedes construir tu propio perfil pasando arrays de presion, temperatura y VMR por gas.

In [ ]:
profile = us_standard_atmosphere(gases=["CO2"])

# Inspeccionar las capas (bloques de integracion hidrostatics)
layers = profile.build_layers()
print(f"Capas     : {len(layers.pressure_atm)}")
print(f"Tope      : {layers.pressure_atm[0] * 1013.25:.3f} hPa")
print(f"Superficie: {layers.pressure_atm[-1] * 1013.25:.2f} hPa")
print(f"CO2 VMR sup: {layers.vmr['CO2'][-1]:.2e}")

---
## Opcion A — run_forward con YAML

Edita `forward_config.yaml` en la raiz del repo para controlar el rango espectral,
la resolucion, el angulo de vision y los outputs que quieres.

In [ ]:
# Ver el contenido del YAML que se va a usar
import pathlib
print(pathlib.Path("forward_config.yaml").read_text(encoding="utf-8"))

In [ ]:
result = run_forward(sd, profile, "forward_config.yaml")
print(result)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(result.wavenumber_cm1, result.transmittance, lw=0.6, color="steelblue")
ax.set_xlabel("Numero de onda [cm$^{-1}$]")
ax.set_ylabel("Transmitancia")
ax.set_title("CO2 iso1  |  700-750 cm$^{-1}$  |  US Standard Atmosphere")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### Guardar resultados a disco

Activa `output.save.enabled: true` en el YAML para guardado automatico, o llama `.save()` manualmente:

In [ ]:
# result.save()                          # usa path/format/prefix del YAML
# result.save(path="out/", fmt="csv")    # override manual

---
## Opcion B — compute_transmittance (API directa)

Sin YAML — util para exploración rapida o cuando parametrizas desde Python.

In [ ]:
nu, T = compute_transmittance(
    sd,
    profile,
    nu_min_cm1=710.0,
    nu_max_cm1=740.0,
    step_cm1=0.005,   # mas rapido para prototipado
    mu_view=1.0,
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(nu, T, lw=0.6, color="darkorange")
ax.set_xlabel("Numero de onda [cm$^{-1}$]")
ax.set_ylabel("Transmitancia")
ax.set_title("CO2 iso1  |  710-740 cm$^{-1}$  |  step=0.005 cm$^{-1}$")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## Perfil propio (opcional)

Si tienes tus propios arrays de presion y temperatura, construye el `AtmosphericProfile` directamente.

In [ ]:
from pylblrtm import AtmosphericProfile
import numpy as np

# Perfil de 3 niveles (TOA primero, superficie ultimo)
mi_perfil = AtmosphericProfile(
    pressure_hpa   = [10.0,  100.0, 1013.25],  # [hPa], TOA → superficie
    temperature_k  = [220.0, 265.0,  288.0],
    vmr            = {"CO2": [4.15e-4] * 3},    # fraccion molar volumetrica
)

nu2, T2 = compute_transmittance(sd, mi_perfil, 710, 740, step_cm1=0.005)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(nu2, T2, lw=0.6, color="seagreen")
ax.set_xlabel("Numero de onda [cm$^{-1}$]")
ax.set_ylabel("Transmitancia")
ax.set_title("CO2 iso1  |  perfil propio 3 capas")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()